In [2]:
# ==============================================================================
# PHẦN 0: CÀI ĐẶT MÔI TRƯỜNG & THƯ VIỆN
# ==============================================================================
!pip install "numpy<2.0"
!pip install -q transformers datasets sentencepiece psutil opendatasets accelerate peft

import os
import time
import shutil
import json
import torch
import gc
import nltk
import opendatasets as od
from google.colab import drive
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback
)
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType

torch.cuda.empty_cache()
gc.collect()

# ==============================================================================
# PHẦN 1: SETUP & TẢI DỮ LIỆU
# ==============================================================================
print(">>> [1/6] Setup Môi trường & Drive...")
drive.mount('/content/drive')

FINAL_SAVE_PATH = "/content/drive/My Drive/T5_Small_Spider_LoRA"
CHECKPOINT_DIR = "/content/drive/My Drive/T5_Small_Spider_LoRA/checkpoints"

print(">>> [2/6] Tải dữ liệu Spider...")
if os.path.exists('spider_data_raw'): shutil.rmtree('spider_data_raw')
if os.path.exists('spider'): shutil.rmtree('spider')

# Nhập Kaggle User/Key tại đây
od.download("https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset")

source_path = "yale-universitys-spider-10-nlp-dataset/spider"
if os.path.exists(source_path):
    shutil.move(source_path, "spider_data_raw")
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
    !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
    nltk.download('punkt')
    nltk.download('punkt_tab')
else:
    raise ValueError("❌ Lỗi tải dữ liệu!")

# ==============================================================================
# PHẦN 2: PREPROCESSING
# ==============================================================================
print("\n>>> [3/6] Xử lý Schema...")

MODEL_NAME = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

with open('spider_data_raw/tables.json', 'r') as f:
    table_data = json.load(f)

db_schema_raw = {}
for db in table_data:
    tables = []
    for table_idx, table_name in enumerate(db['table_names_original']):
        cols = [c[1] for c in db['column_names_original'] if c[0] == table_idx]
        tables.append({"name": table_name, "cols": cols})
    db_schema_raw[db['db_id']] = tables

def get_serialized_schema(question, db_id):
    if db_id not in db_schema_raw: return ""
    tables = db_schema_raw[db_id]
    relevant_tables = []
    question_lower = question.lower()
    for tbl in tables:
        if tbl['name'].lower() in question_lower:
            relevant_tables.append(tbl)
    if not relevant_tables:
        relevant_tables = tables[:6]

    schema_parts = [f"Table: {tbl['name']} | Columns: {', '.join(tbl['cols'])}" for tbl in relevant_tables]
    return " || ".join(schema_parts)

def preprocess_function(examples):
    inputs = []
    targets = []
    for i in range(len(examples['question'])):
        question = examples['question'][i]
        db_id = examples['db_id'][i]
        schema = get_serialized_schema(question, db_id)
        inputs.append(f"translate to SQL: {question} | Schema: {schema}")
        targets.append(examples['query'][i])

    model_inputs = tokenizer(inputs, max_length=384, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=128, padding="max_length", truncation=True)
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset = load_dataset("spider")
train_data = dataset["train"].map(preprocess_function, batched=True)
val_data = dataset["validation"].map(preprocess_function, batched=True)

# ==============================================================================
# PHẦN 3: TRAINING (Đã sửa lỗi NameError & Cấu hình LoRA tối ưu)
# ==============================================================================
from transformers import TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType

print(f"\n>>> [4/6] Bắt đầu Training {MODEL_NAME} với LoRA (Optimized)...")

# --- 1. Load Base Model ---
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# --- 2. Cấu hình LoRA (Phiên bản mạnh: r=64, target all modules) ---
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=64,               # Rank cao để học tốt hơn
    lora_alpha=64,      # Alpha = Rank
    lora_dropout=0.05,
    # Target tất cả các lớp quan trọng trong T5
    target_modules=["q", "v", "k", "o", "wi", "wo"],
    bias="none"
)

model = get_peft_model(model, peft_config)
print("📊 Số lượng tham số LoRA:")
model.print_trainable_parameters()

# --- 3. Định nghĩa Callback (SỬA LỖI NAME ERROR TẠI ĐÂY) ---
class EfficiencyCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("\n📊 Bắt đầu theo dõi tài nguyên...")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if torch.cuda.is_available():
            # Lấy thông tin VRAM
            max_mem = torch.cuda.max_memory_allocated() / (1024**3)
            logs["max_vram_gb"] = round(max_mem, 2)

    def on_train_end(self, args, state, control, **kwargs):
        duration = time.time() - self.start_time
        hours = duration // 3600
        mins = (duration % 3600) // 60
        print(f"\n⏱️ Tổng thời gian train: {int(hours)}h {int(mins)}m")
        if torch.cuda.is_available():
            print(f"💾 VRAM tiêu thụ tối đa: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")

# --- 4. Training Arguments (Đã sửa warmup_ratio -> warmup_steps) ---
training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    group_by_length=True,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    fp16=True,

    learning_rate=1e-3,
    optim="adafactor",
    max_grad_norm=1.0,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    report_to="none"
)

# --- 5. Khởi tạo Trainer ---
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    callbacks=[EfficiencyCallback()]  # Class đã được định nghĩa ở trên
)

# --- 6. Chạy Training ---
# Kiểm tra checkpoint cũ
last_checkpoint = get_last_checkpoint(CHECKPOINT_DIR)
if last_checkpoint is not None:
    print(f"🔄 Đang khôi phục từ checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=True)
else:
    trainer.train()

# --- 7. Lưu Model ---
trainer.save_model(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)
print(f"✅ Đã lưu LoRA adapters tại {FINAL_SAVE_PATH}")

# ==============================================================================
# PHẦN 4: INFERENCE (Đã thêm Merge để tăng tốc)
# ==============================================================================
print("\n>>> [5/6] Sinh SQL & Đo lường độ trễ (Latency)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# --- PHẦN QUAN TRỌNG: MERGE LO-RA VÀO MODEL GỐC ---
print("🔄 Đang gộp (Merge) LoRA adapter vào model gốc để tối ưu tốc độ...")
model = model.merge_and_unload()
# --------------------------------------------------

predictions = []
gold_lines = []
total_inference_time = 0

print(f"📝 Đang chạy inference trên {len(dataset['validation'])} mẫu...")

for i, item in enumerate(dataset["validation"]):
    schema = get_serialized_schema(item['question'], item['db_id'])
    input_ids = tokenizer(f"translate to SQL: {item['question']} | Schema: {schema}", return_tensors="pt", max_length=384, truncation=True).input_ids.to(device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    total_inference_time += (time.time() - start)

    predictions.append(tokenizer.decode(outputs[0], skip_special_tokens=True) + "\n")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

    if (i+1) % 200 == 0:
        print(f"   ...Đã xong {i+1}/{len(dataset['validation'])}")

print(f"\n⚡ Efficiency Report ({MODEL_NAME} + LoRA Merged):")
print(f"   Avg Latency: {(total_inference_time / len(dataset['validation'])) * 1000:.2f} ms/sample")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

# ==============================================================================
# PHẦN 5: CHẤM ĐIỂM
# ==============================================================================
print("\n>>> [6/6] Chấm điểm Exact Match & Execution Accuracy...")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py

db_path = "spider_data_raw/database"
table_path = "spider_data_raw/tables.json"

if os.path.exists(db_path):
    !python evaluation.py --gold gold.txt --pred pred.txt --db {db_path} --table {table_path} --etype all
else:
    print("❌ Lỗi database.")

>>> [1/6] Setup Môi trường & Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
>>> [2/6] Tải dữ liệu Spider...
Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: phankhaclap
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset


100%|██████████| 96.0M/96.0M [00:00<00:00, 722MB/s]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



>>> [3/6] Xử lý Schema...


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

...


>>> [4/6] Bắt đầu Training t5-small với LoRA (Optimized)...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

📊 Số lượng tham số LoRA:
trainable params: 8,650,752 || all params: 69,157,376 || trainable%: 12.5088

📊 Bắt đầu theo dõi tài nguyên...


Epoch,Training Loss,Validation Loss,Vram Gb
1,1.961555,0.950424,1.340000
2,1.313565,0.833127,1.340000
3,1.005935,0.815084,1.340000
4,0.867280,0.816458,1.340000
5,0.769382,0.850270,1.340000
6,0.653199,0.847761,1.340000
7,0.603096,0.864291,1.340000
8,0.528604,0.850968,1.340000
9,0.499130,0.860043,1.340000
10,0.474879,0.898831,1.340000



⏱️ Tổng thời gian train: 0h 55m
💾 VRAM tiêu thụ tối đa: 1.34 GB
✅ Đã lưu LoRA adapters tại /content/drive/My Drive/T5_Small_Spider_LoRA

>>> [5/6] Sinh SQL & Đo lường độ trễ (Latency)...
🔄 Đang gộp (Merge) LoRA adapter vào model gốc để tối ưu tốc độ...
📝 Đang chạy inference trên 1034 mẫu...
   ...Đã xong 200/1034
   ...Đã xong 400/1034
   ...Đã xong 600/1034
   ...Đã xong 800/1034
   ...Đã xong 1000/1034

⚡ Efficiency Report (t5-small + LoRA Merged):
   Avg Latency: 471.95 ms/sample

>>> [6/6] Chấm điểm Exact Match & Execution Accuracy...
medium pred: SELECT Song_Name, Country, Age FROM singer ORDER BY Age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

eval_err_num:1
medium pred: SELECT T2.location, T1.name FROM stadium AS T1 JOIN concert AS T2 ON T1.stadium_id = T2.stadium_id WHERE T2.capacity BETWEEN 5000 AND 10000
medium gold: SELECT LOCATION ,  name FROM stadium WHERE capacity BETWEEN 5000 AND 10000

medium pred: SELECT max(capacity), avg(capacity) 